# WebSockets
Starlette 包含一个WebSocket与 HTTP 请求起类似作用的类，但允许在 websocket 上发送和接收数据。

## WebSocket

签名：`WebSocket(scope, receive=None, send=None)`



In [ ]:
from starlette.websockets import WebSocket


async def app(scope, receive, send):
    websocket = WebSocket(scope=scope, receive=receive, send=send)
    await websocket.accept()
    await websocket.send_text('Hello, world!')
    await websocket.close()

WebSockets 提供了一个映射接口，因此您可以像使用`scope`一样使用它们。     
例如：`websocket['path']`将返回 ASGI 路径。
- `URL`
    - `websocket URL` 的访问方式为`websocket.url`。
    - 该属性实际上是的子类str，并且还公开了可以从 URL 中解析出的所有组件。
    - 例如：`websocket.url.path`，`websocket.url.port`，`websocket.url.scheme`。
- `Headers` : headers公开为不可变、不区分大小写的多字典。
    - 例如：`websocket.headers ['sec-websocket-version']`
- `Query Parameters`
    - 查询参数以不可变的多字典形式公开。
    - 例如：`websocket.query_params['search']`
- `Path Parameters`
    - 路由器路径参数以字典接口的形式公开。
    - 例如：`websocket.path_params['username']`
- `Accepting the connection`
    - `await websocket.accept(subprotocol=None, headers=None)`
- `Sending data`
    - `await websocket.send_text(data)`
    - `await websocket.send_bytes(data)`
    - `await websocket.send_json(data)`
    - 从 0.10.0 版本开始，JSON 消息默认通过文本数据帧发送。用于`websocket.send_json(data, mode="binary")`通过二进制数据帧发送 JSON。
- `Receiving data`
    - `await websocket.receive_text()`
    - `await websocket.receive_bytes()`
    - `await websocket.receive_json()`
    - 可能提升`starlette.websockets.WebSocketDisconnect()`。
    - 从 0.10.0 版本开始，JSON 消息默认通过文本数据框接收。用于`websocket.receive_json(data, mode="binary")`通过二进制数据框接收 JSON。
- `Iterating data`
    - websocket.iter_text()
    - websocket.iter_bytes()
    - websocket.iter_json()
    - 类似于`receive_text`、`receive_bytes`和`receive_json`，但返回异步迭代器。
    ```python
    from starlette.websockets import WebSocket

    async def app(scope, receive, send):
        websocket = WebSocket(scope=scope, receive=receive, send=send)
        await websocket.accept()
        async for message in websocket.iter_text():
            await websocket.send_text(f"Message text was: {message}")
        await websocket.close()
    ```
- Closing the connection
    - `await websocket.close(code=1000, reason=None)`
- Sending and receiving messages
    - 如果需要发送或接收原始 ASGI 消息，则应使用 `websocket.send()` 和 `websocket.receive()`，而不是使用原始发送和接收可调用程序。
    - 这如果要发送不同的错误响应，可以使用 `websocket.send_denial_response（）` 方法。这将发送响应，然后关闭连接。将确保` websocket` 的状态保持正确更新。
        -  `await websocket.send_denial_response(response)`
    - 这要求 ASGI 服务器支持 WebSocket 拒绝响应扩展。如果不支持，将引发 `RuntimeError`。
    - 在 `Starlette` 中，您也可以使用 `HTTPException` 来达到同样的效果。


In [ ]:
from starlette.applications import Starlette
from starlette.exceptions import HTTPException
from starlette.routing import WebSocketRoute
from starlette.websockets import WebSocket


def is_authorized(subprotocols: list[str]):
    if len(subprotocols) != 2:
        return False
    if subprotocols[0] != "Authorization":
        return False
    # Here we are hard coding the token, in a real application you would validate the token
    # against a database or an external service.
    if subprotocols[1] != "token":
        return False
    return True


async def websocket_endpoint(websocket: WebSocket):
    subprotocols = websocket.scope["subprotocols"]
    if not is_authorized(subprotocols):
        raise HTTPException(status_code=401, detail="Unauthorized")
    await websocket.accept("Authorization")
    await websocket.send_text("Hello, world!")
    await websocket.close()


app = Starlette(debug=True, routes=[WebSocketRoute("/ws", websocket_endpoint)])